|        |        |        |
|--------|--------|--------|
![H-BRS](logos/h-brs.png) | ![A2S](logos/a2s.png) | ![b-it](logos/b-it.png) |

# Software Engineering for Robotics

# SER Assignment 1

#### Team members:
* prai2s
* mhossa2s
* ...
YOUR ANSWER HERE

# Robot Behaviour Management Using State Machines and Behaviour Trees

In this assignment, we will compare the implementation of behaviour trees and state machines to establish primary safety features for a robot; this includes situations such as the battery level falling below a certain threshold and avoiding potential collisions. In other words, we want to implement both a behaviour tree and a state machine to achieve the same functionality.

We particularly want the robot to behave as follows:
* *The battery level falling below a given threshold*: The robot starts rotating in place until the level is above the threshold again (**you can control the battery level by publishing to a topic you define**).
* *A collision is about to happen*: The robot stops moving and needs to be moved to a safe distance manually.

**You will use the Robile simulation for testing your implementation. For your submission, you need to add your code to the appropriate cells below; however, note that, to actually test your implementation, you need to integrate the code in a ROS package and perform all tests on your local machine.**

## Robot Safety Functionalities Using a State Machine [45 points]

To implement a state machine, we will use the [SMACH framework](https://wiki.ros.org/smach/Tutorials). SMACH as such is ROS independent, but `executive_smach` provides a ROS integration, which we will be using in this assignment.

Your task here is to implement the necessary states and set up the state machine to achieve the desired robot functionality.

In [ ]:
import rclpy
import smach

from std_msgs.msg import Float32
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist


# Monitor
class MonitorBatteryAndCollision(smach.State):

    def __init__(self, node):

        self.node = node

        self.node.battery = 100.0
        self.battery_threshold = 30.0
        self.obstacle_close = False

        self.battery_sub = node.create_subscription(
            Float32,
            '/battery_voltage',
            self.battery_callback,
            10
        )

        self.scan_sub = node.create_subscription(
            LaserScan,
            '/scan',
            self.scan_callback,
            10
        )

        smach.State.__init__(
            self,
            outcomes=['battery_low', 'collision', 'safe']
        )

    def battery_callback(self, msg):
        self.node.battery = msg.data

    def scan_callback(self, msg):
        if len(msg.ranges) > 0:
            self.obstacle_close = min(msg.ranges) < 0.3

    def execute(self, userdata):

        rclpy.spin_once(self.node, timeout_sec=0.1)

        battery = self.node.battery  # FIXED usage

        if self.obstacle_close:
            self.node.get_logger().info("COLLISION detected")
            return 'collision'

        if battery < self.battery_threshold:
            self.node.get_logger().info("BATTERY LOW")
            return 'battery_low'

        self.node.get_logger().info("SAFE state")
        return 'safe'


# Rotate
class RotateBase(smach.State):

    def __init__(self, node):

        self.node = node

        self.pub = node.create_publisher(
            Twist,
            '/cmd_vel',
            10
        )

        smach.State.__init__(
            self,
            outcomes=['battery_ok', 'rotate']
        )

    def execute(self, userdata):

        self.node.get_logger().info("ROTATING... low battery")

        msg = Twist()
        msg.angular.z = 1.0
        self.pub.publish(msg)

        rclpy.spin_once(self.node, timeout_sec=0.1)

        battery = self.node.battery

        if battery >= 30:
            return 'battery_ok'

        return 'rotate'


# Stop
class StopMotion(smach.State):

    def __init__(self, node):

        self.node = node

        self.pub = node.create_publisher(
            Twist,
            '/cmd_vel',
            10
        )

        smach.State.__init__(
            self,
            outcomes=['clear']
        )

    def execute(self, userdata):

        self.node.get_logger().info("STOPPING robot (collision)")

        msg = Twist()
        msg.linear.x = 0.0
        msg.angular.z = 0.0

        self.pub.publish(msg)

        rclpy.spin_once(self.node, timeout_sec=0.1)

        return 'clear'


# Main
def main(args=None):

    rclpy.init(args=args)

    node = rclpy.create_node("smach_state_machine")

    sm = smach.StateMachine(outcomes=['DONE'])

    with sm:

        smach.StateMachine.add(
            'MONITOR',
            MonitorBatteryAndCollision(node),
            transitions={
                'battery_low': 'ROTATE',
                'collision': 'STOP',
                'safe': 'MONITOR'
            }
        )

        smach.StateMachine.add(
            'ROTATE',
            RotateBase(node),
            transitions={
                'battery_ok': 'MONITOR',
                'rotate': 'ROTATE'
            }
        )

        smach.StateMachine.add(
            'STOP',
            StopMotion(node),
            transitions={
                'clear': 'MONITOR'
            }
        )

    sm.execute()

    rclpy.spin(node)


if __name__ == "__main__":
    main()

## Robot Safety Functionalities Using a Behaviour Tree [45 points]

The majority of implementations of behaviour trees in robotics are using `BehaviorTree.CPP` in cpp and [py_trees](https://py-trees.readthedocs.io/en/devel/) in Python. [py_trees_ros](https://py-trees-ros-tutorials.readthedocs.io/en/release-2.1.x/tutorials.html) is a wrapper for `py_trees` to integrate it with ROS, which we will use in this assignment.

Your task here is to implement the necessary behaviours and set up the behaviour tree to achieve the desired robot functionality.

Implement the required behaviours in the cell below. [35 points]

In [ ]:
import rclpy
import py_trees as pt
import py_trees_ros as ptr
from geometry_msgs.msg import Twist
from std_msgs.msg import Float32
from sensor_msgs.msg import LaserScan
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSHistoryPolicy


class Rotate(pt.behaviour.Behaviour):
    """Rotates the robot about the z-axis
    """

    def __init__(self, name="rotate platform",
                 topic_name="/cmd_vel",
                 ang_vel=1.0):

        super(Rotate, self).__init__(name)

        self.topic_name = topic_name
        self.ang_vel = ang_vel
        self.publisher = None
        self.node = None

    def setup(self, **kwargs):

        self.logger.info("[ROTATE] setting up rotate behaviour")

        try:
            self.node = kwargs['node']
        except KeyError as e:
            error_message = "didn't find 'node' in setup's kwargs"
            raise KeyError(error_message) from e

        self.publisher = self.node.create_publisher(
            Twist,
            self.topic_name,
            10
        )

        return True

    def update(self):

        self.logger.info("[ROTATE] update: updating rotate behaviour")

        msg = Twist()
        msg.angular.z = self.ang_vel

        self.publisher.publish(msg)

        return pt.common.Status.RUNNING

    def terminate(self, new_status):

        self.logger.info("[ROTATE] terminate")

        stop_msg = Twist()
        self.publisher.publish(stop_msg)

        return super().terminate(new_status)


class StopMotion(pt.behaviour.Behaviour):
    """Stops the robot when it is controlled using joystick or cmd_vel
    """

    def __init__(self,
                 name="stop motion",
                 topic_name="/cmd_vel"):

        super().__init__(name)

        self.topic_name = topic_name
        self.publisher = None
        self.node = None

    def setup(self, **kwargs):

        self.node = kwargs['node']

        self.publisher = self.node.create_publisher(
            Twist,
            self.topic_name,
            10
        )

        return True

    def update(self):

        msg = Twist()
        msg.linear.x = 0.0
        msg.angular.z = 0.0

        self.publisher.publish(msg)

        return pt.common.Status.SUCCESS

    def terminate(self, new_status):

        stop_msg = Twist()
        self.publisher.publish(stop_msg)

        return super().terminate(new_status)


class BatteryStatus2bb(ptr.subscribers.ToBlackboard):
    """Checks the battery status
    """

    def __init__(self,
                 battery_voltage_topic_name="/battery_voltage",
                 name='Battery2BB',
                 threshold=30.0):

        super().__init__(
            name=name,
            topic_name=battery_voltage_topic_name,
            topic_type=Float32,
            blackboard_variables={'battery': 'data'},
            initialise_variables={'battery': 100.0},
            clearing_policy=pt.common.ClearingPolicy.NEVER,
            qos_profile=ptr.utilities.qos_profile_unlatched()
        )

        self.threshold = threshold

        self.blackboard.register_key(
            key='battery_low_warning',
            access=pt.common.Access.WRITE
        )

    def update(self):

        self.logger.info('[BATTERY] update')

        super().update()

        battery = self.blackboard.battery

        self.blackboard.battery_low_warning = (
            battery < self.threshold
        )

        return pt.common.Status.SUCCESS


class LaserScan2bb(ptr.subscribers.ToBlackboard):
    """Checks the laser scan measurements to avoid possible collisions.
    """

    def __init__(self,
                 topic_name="/scan",
                 name='Scan2BB',
                 safe_range=0.25):

        super().__init__(
            name=name,
            topic_name=topic_name,
            topic_type=LaserScan,
            blackboard_variables={'laser_scan': 'ranges'},
            clearing_policy=pt.common.ClearingPolicy.NEVER,
            qos_profile=QoSProfile(
                reliability=QoSReliabilityPolicy.RMW_QOS_POLICY_RELIABILITY_BEST_EFFORT,
                history=QoSHistoryPolicy.RMW_QOS_POLICY_HISTORY_KEEP_LAST,
                depth=10
            )
        )

        self.safe_range = safe_range

        self.blackboard.register_key(
            key='collision',
            access=pt.common.Access.WRITE
        )

    def update(self):

        super().update()

        ranges = self.blackboard.laser_scan

        valid_ranges = [r for r in ranges if r > 0.0]

        if len(valid_ranges) > 0:

            min_distance = min(valid_ranges)

            self.blackboard.collision = (
                min_distance < self.safe_range
            )

        else:
            self.blackboard.collision = False

        return pt.common.Status.SUCCESS

Now, set up and initialise your behaviour tree in the cell below. [10 points]

In [ ]:
import py_trees as pt
import py_trees_ros as ptr
import operator

import py_trees.console as console
import rclpy
import sys


def create_root() -> pt.behaviour.Behaviour:
    """Structures a behaviour tree to monitor the battery status,
    rotate if battery is low, and stop if obstacle detected.
    """

    root = pt.composites.Parallel(
        name="root",
        policy=pt.common.ParallelPolicy.SuccessOnAll(
            synchronise=False
        )
    )

    topics2BB = pt.composites.Sequence(
        "Topics2BB",
        memory=False
    )

    priorities = pt.composites.Selector(
        "Priorities",
        memory=False
    )

    idle = pt.behaviours.Running(name="Idle")

    # Sensor behaviours
    battery_sensor = BatteryStatus2bb()
    laser_sensor = LaserScan2bb()

    topics2BB.add_children([
        battery_sensor,
        laser_sensor
    ])

    rotate = Rotate()
    stop = StopMotion()

    battery_low = pt.behaviours.CheckBlackboardVariableValue(
        name="Battery Low?",
        check=pt.common.ComparisonExpression(
            variable="battery_low_warning",
            value=True,
            operator=operator.eq
        )
    )

    collision_detected = pt.behaviours.CheckBlackboardVariableValue(
        name="Collision?",
        check=pt.common.ComparisonExpression(
            variable="collision",
            value=True,
            operator=operator.eq
        )
    )

    collision_handler = pt.composites.Sequence(
        name="Collision Handler",
        memory=False
    )

    collision_handler.add_children([
        collision_detected,
        stop
    ])

    battery_handler = pt.composites.Sequence(
        name="Battery Handler",
        memory=False
    )

    battery_handler.add_children([
        battery_low,
        rotate
    ])

    
    priorities.add_children([
        collision_handler,
        battery_handler,
        idle
    ])

    root.add_children([
        topics2BB,
        priorities
    ])

    return root


def main():

    rclpy.init(args=None)

    root = create_root()

    tree = ptr.trees.BehaviourTree(
        root=root,
        unicode_tree_debug=True
    )

    try:
        tree.setup(timeout=20.0)

    except ptr.exceptions.TimedOutError as e:
        console.logerror(
            console.red +
            "failed to setup the tree [{}]".format(str(e)) +
            console.reset
        )

        tree.shutdown()
        rclpy.try_shutdown()
        sys.exit(1)

    pt.display.render_dot_tree(root)

    tree.tick_tock(period_ms=200)

    try:
        rclpy.spin(tree.node)

    except (
        KeyboardInterrupt,
        rclpy.executors.ExternalShutdownException
    ):
        pass

    finally:
        tree.shutdown()
        rclpy.try_shutdown()


if __name__ == '__main__':
    main()

## Setting up Your System for Testing

1. (If not done already) Please set up Ubuntu 22.04, ROS2 humble, and the Robile simulation by following the [documentation](https://robile-amr.readthedocs.io/en/humble/).

2. Clone the `executive_smach` repository in the src folder of your workspace, and also install the `py-trees-ros` package:
```
cd ~/ros2_ws/src/  
git clone -b ros2 https://github.com/ros/executive_smach.git  
sudo apt-get install ros-humble-py-trees ros-humble-py-trees-ros ros-humble-py-trees-ros-interfaces xcb
```

From the workspace directory, build the workspace:
```
cd ~/ros2_ws/
colcon build --symlink-install
```

Now source the workspace setup file:
```
source install/setup.bash
```
3. Create a new ROS2 python package and integrate your implementation in it

## Testing Instructions

Use the following steps to test your implementation:
- Run the Robile in simulation
- After integrating your implementation in your local ROS workspace, launch your components to test the functionalities. **Note that you need to test the state machine and behaviour tree implementations independently to verify that the robot behaves exactly the same in both cases.**

**In your submission, please include screenshots to verify your implementation, and link them in the cell further below.**

As already mentioned before, as the battery percentage is not readily available in simulation, please publish the battery percentage values manually. For instance, if the topic `/battery_voltage` is used for monitoring the battery status, you should be able to publish a battery percentage value of your choice to verify your implementation, e.g.:
```  
ros2 topic  pub /battery_voltage std_msgs/msg/Float32 "data: 50.0"
```

Finally, behaviour tree visualization is not released on ROS2 humble yet, but the changes in the behaviour tree can be monitored by running the following command, which is helpful for debugging:
```
py-trees-tree-watcher
```

The following is a sample visualisation when the robot is about to collide:

![collision avoidance BT](figures/BT_watcher.png)

For getting a better intuition of behaviour trees, another sample visualisation of a similar task, but with a slightly different structure can be found below:

![collison and battery low](figures/collision_battery.png)

**Discuss any observations from your tests and include screenshots that verify your implementation in the cell below. [10 points]**

The finite state machine and behaviour tree implementations were successfully tested in the Gazebo simulation environment. The robot correctly monitored battery voltage and laser scan data using ROS2 topics. When the battery voltage dropped below the defined threshold, the robot started rotating as expected. Similarly, when an obstacle was detected within the safe laser scan range, the collision handling behaviour was triggered and the robot stopped successfully. The behaviour tree also correctly prioritised collision avoidance over low battery behaviour, demonstrating proper safety handling and decision-making. The screen-recorded videos and the state machine and behaviour tree diagrams have been uploaded to the following Google Drive folder: https://drive.google.com/drive/folders/1OsGcZoLbWo0MDvVpxx_V-3y8H_Bz2c0c?usp=sharing